In [25]:
# Group 9

# Imports Relevant Packages
import numpy as np
import pandas as pd
from pyomo.environ import *

# Read Excel File & Create Dataframe 
file_name = 'Group_Assignment_Dataset.xlsx'
df = pd.read_excel(file_name, 'Data', header=None)

# Extract Primary Inputs
fleet = df.iloc[2, 1]                                                 # Total Fleet Size
initial_inventory = df.iloc[3, 1]                                     # Initial Inventory in Week 1
unit_cost = df.iloc[4, 1]                                             # Unit Cost (£) per Generator
durations = [int(x.strip()) for x in str(df.iloc[5, 1]).split(',')]   # Rental Durations

# A List to Store Cleaned Data
cleaned_data = []

# Reads Weekly Data Rows (from Weeks 1 - 52)
for i in range(10, 62): 
    week = df.iloc[i, 1]   #Extracts the Total Number of Weeks

    # Stores All Values in Dictionary
    cleaned_data.append({'Week': week, 'Date': df.iloc[i, 0], 'Inventory': df.iloc[i, 2],
                         'Price_1': df.iloc[i, 4], 'Demand_1': df.iloc[i, 5], 'Accept_1': df.iloc[i, 6], 'Return_1': df.iloc[i, 7],
                         'Price_4': df.iloc[i, 8], 'Demand_4': df.iloc[i, 9], 'Accept_4': df.iloc[i, 10], 'Return_4': df.iloc[i, 11],
                         'Price_8': df.iloc[i, 12], 'Demand_8': df.iloc[i, 13], 'Accept_8': df.iloc[i, 14], 'Return_8': df.iloc[i, 15],
                         'Price_16': df.iloc[i, 16], 'Demand_16': df.iloc[i, 17], 'Accept_16': df.iloc[i, 18], 'Return_16': df.iloc[i, 19]})

# Converts List of Dictionaries to Dataframe Indexed by Week
df = pd.DataFrame(cleaned_data).set_index('Week')

# Weeks used in Optimisation (Weeks 1 and 52 excluded)
weeks = [i for i in df.index if 2 <= i <= 51]

numofweeks = len(weeks)
numofdurations = len(durations)


# Opening Inventory for Each Week
opening_inventory = df.loc[weeks, 'Inventory'].to_numpy()


# Numpy Arrays for Price, Demand, and Actual Rentals
price_np = np.array([[df.loc[i, 'Price_' + str(j)] for j in durations] for i in weeks])
demand_np = np.array([[df.loc[i, 'Demand_' + str(j)] for j in durations] for i in weeks])
actual_np = np.array([[df.loc[i, 'Accept_' + str(j)] for j in durations] for i in weeks])


# Function to Retrieve Accepted Rentals from Previous Weeks
def get_actual_acc(week_lag, duration):
    if week_lag in df.index and week_lag >= 2:
        return df.loc[week_lag, 'Accept_' + str(duration)]
    else:
         return 0

# Calculates Additional Returns Each Week
pre_returns = []

for i in weeks:
    extra_returns = 0
    for j in durations:
        extra_returns = extra_returns + max(0, df.loc[i, 'Return_' + str(j)] - get_actual_acc(i - j, j))
    pre_returns.append(extra_returns)

# Calculates Bw (Available Inventory Each Week)
Bw = []
running_returns = 0

for i in range(numofweeks):
    running_returns = running_returns + pre_returns[i]
    Bw.append(int(initial_inventory + running_returns))

# Create Model
model = ConcreteModel()

# Decision Variable
model.x = Var(range(numofweeks), range(numofdurations), domain=NonNegativeReals)

# Objective Function
def model_objective(model):
    return sum(7 * durations[j] * price_np[i, j] * model.x[i, j] for i in range(numofweeks) for j in range(numofdurations))

model.obj = Objective(rule=model_objective, sense=maximize)

# Demand Constraint (Cannot Accept More Rentals than Demand)
def demand_rule(model, i, j):
    return model.x[i, j] <= demand_np[i, j]

model.cost_demand = Constraint(range(numofweeks), range(numofdurations), rule=demand_rule)

# Capacity Constraint (Inventory Available Each Week)
def cap_rule(model, w):
    return sum(model.x[i, j] for i in range(numofweeks) for j in range(numofdurations) if i <= w and i + durations[j] > w) <= Bw[w]

model.cost_cap = Constraint(range(numofweeks), rule=cap_rule)

# Print Main Parameters
print(f'Total Fleet: {fleet} generators')
print(f'Initial Inventory: {initial_inventory} generators')
print(f'Unit Cost: £{unit_cost}')
print('\n'*2)

# Solve Optimisation Model
solver = SolverFactory('glpk')
results = solver.solve(model)

## Revenue
print('REVENUE')
print()

actual_revenue = 0
for i in range(numofweeks):
    for j in range(numofdurations):
        actual_revenue = actual_revenue + 7 * durations[j] * price_np[i, j] * actual_np[i, j]

if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    print(f"Total Actual Revenue: £{actual_revenue:,.2f}")
    print(f"Total Optimised Revenue: £{value(model.obj()):,.2f}")
    print('\n'*2)

## Optimal Inventory Path: Bw derived
# I_opt[w] = Bw[w] - Σ x_{t,k} (t < w, t+k > w) : inventory before accpecting
I_opt = {1: initial_inventory}
for w_idx, w in enumerate(weeks):
    lp_out = sum(value(model.x[i, j]) for i in range(numofweeks) for j in range(numofdurations) if weeks[i] < w and weeks[i] + durations[j] > w)
    I_opt[w] = Bw[w_idx] - lp_out

# Week 52: R_52 = NaN → processing this as 0
acc_51    = sum(value(model.x[weeks.index(51), j]) for j in range(numofdurations))
I_opt[52] = max(0, I_opt[51] - acc_51)

## Load Factor
print('LOAD FACTOR')
print()

actual_load_factor = np.mean((fleet - df.loc[range(1, 53), 'Inventory'].to_numpy()) / fleet) 
optimised_load_factor = np.mean([(fleet - I_opt[i]) / fleet for i in range(1, 53)])

print(f'Actual Load Factor: {round(actual_load_factor*100,2)}%')
print(f'Optimised Load Factor: {round(optimised_load_factor*100,2)}%')
print('\n'*2)

## Return on Investment (ROI) 
print('RETURN ON INVESTMENT (ROI)')
print()

actual_roi = actual_revenue / (fleet * unit_cost)
optimised_roi = value(model.obj()) / (fleet * unit_cost)

print('Actual ROI:', round(actual_roi,2))
print('Optimised ROI:', round(optimised_roi,2))
print('\n'*2)

## Shadow Prices
print("SHADOW PRICES")
print()

initial_revenue = value(model.obj())

for w in range(numofweeks):
    Bw_new = Bw.copy()
    Bw_new[w] = Bw_new[w] + 1

    model_sp = ConcreteModel()
    
    model_sp.x = Var(range(numofweeks), range(numofdurations), domain=NonNegativeReals)

    model_sp.obj = Objective(expr=sum(7 * durations[j] * price_np[i, j] * model_sp.x[i, j] for i in range(numofweeks) for j in range(numofdurations)), sense=maximize)

    model_sp.cost_demand = Constraint(range(numofweeks), range(numofdurations), rule=lambda m,i,j: m.x[i,j] <= demand_np[i,j])

    model_sp.cost_cap = Constraint(range(numofweeks), rule=lambda m,k: sum(m.x[i,j] for i in range(numofweeks) for j in range(numofdurations) if i <= k and i + durations[j] > k) <= Bw_new[k])

    solver.solve(model_sp)

    shadow_price = value(model_sp.obj()) - initial_revenue

    if shadow_price > 0:
        print(f"Week {weeks[w]} Shadow Price: £{round(shadow_price,2)}")
        
print('\n'*2)

# Supplementary Breakdown by Rental Duration
print('REVENUE BREAKDOWN BY RENTAL DURATION (ACTUAL VS. OPTIMISED)')
print()
rows = []
for j, k in enumerate(durations):
    actual_revenue = sum(7*k*price_np[i,j]*actual_np[i,j] for i in range(numofweeks))
    optimal_revenue = sum(7*k*price_np[i,j]*value(model.x[i,j]) for i in range(numofweeks))

    rows.append({'Duration': f'{k}-week', 
        'Actual Requests Accepted': int(sum(actual_np[i, j] for i in range(numofweeks))),
        'Optimal Requests Accepted': int(sum(value(model.x[i, j]) for i in range(numofweeks))),
        'Actual Revenue': f'£{actual_revenue:,.2f}',
        'Optimised Revenue': f'£{optimal_revenue:,.2f}'})

summary = pd.DataFrame(rows).set_index('Duration')
print(summary.to_string())

Total Fleet: 300 generators
Initial Inventory: 77 generators
Unit Cost: £3000



REVENUE

Total Actual Revenue: £4,650,478.70
Total Optimised Revenue: £5,116,909.00



LOAD FACTOR

Actual Load Factor: 73.38%
Optimised Load Factor: 79.12%



RETURN ON INVESTMENT (ROI)

Actual ROI: 5.17
Optimised ROI: 5.69



SHADOW PRICES

Week 24 Shadow Price: £313.6
Week 25 Shadow Price: £168.7
Week 26 Shadow Price: £541.1
Week 27 Shadow Price: £555.8
Week 28 Shadow Price: £84.0
Week 29 Shadow Price: £572.6
Week 30 Shadow Price: £544.6
Week 31 Shadow Price: £534.8
Week 32 Shadow Price: £204.4
Week 36 Shadow Price: £512.4
Week 37 Shadow Price: £672.7
Week 38 Shadow Price: £2.8
Week 39 Shadow Price: £72.8
Week 40 Shadow Price: £512.4



REVENUE BREAKDOWN BY RENTAL DURATION (ACTUAL VS. OPTIMISED)

          Actual Requests Accepted  Optimal Requests Accepted Actual Revenue Optimised Revenue
Duration                                                                                      
1-week              

In [26]:
# Inventory Comparison
acc_actual = {(i, k): df.loc[i, f'Accept_{k}'] for i in range(1, 53) for k in durations}
total_acc_actual = {i: sum(acc_actual[(i, k)] for k in durations) for i in range(1, 53)}

# Per-duration LP acceptances and returns
lp_acc = {(i, k): value(model.x[weeks.index(i), durations.index(k)]) if i in weeks else 0.0 for i in range(1, 53) for k in durations}

total_acc_lp = {i: sum(value(model.x[weeks.index(i), j]) for j in range(numofdurations)) if i in weeks else 0.0 for i in range(1, 53)}

ret = {(i, k): df.loc[i, f'Return_{k}'] for i in range(1, 53) for k in durations}

total_ret = {i: sum(df.loc[i, f'Return_{k}'] for k in durations) for i in range(1, 53)}

demand_by_dur = {(i, k): float(df.loc[i, f'Demand_{k}']) for i in range(1, 53) for k in durations}

total_demand = {i: sum(demand_by_dur[(i, k)] for k in durations) for i in range(1, 53)}

# Build DataFrame
data = {}
data['Act_Op_Inv'] = [df.loc[w, 'Inventory'] for w in range(1, 53)]
data['LP_Op_Inv'] = [I_opt[w] for w in range(1, 53)]
data['LP_Acc'] = [total_acc_lp[w] for w in range(1, 53)]
data['Act_Acc'] = [total_acc_actual[w] for w in range(1, 53)]
data['Tot_Ret'] = [total_ret[w] for w in range(1, 53)]
data['Tot_Dem'] = [total_demand[w] for w in range(1, 53)]

for k in durations:
    data[f'LP-Acc-{k}-wk'] = [lp_acc[(w, k)] for w in range(1, 53)]
    data[f'Act-Acc-{k}-wk']  = [acc_actual[(w, k)] for w in range(1, 53)]
    data[f'Ret-{k}-wk'] = [ret[(w, k)] for w in range(1, 53)]
    data[f'Dem-{k}-wk'] = [demand_by_dur[(w, k)] for w in range(1, 53)]

# Column order
ordered = ['Act_Op_Inv', 'LP_Op_Inv', 'LP_Acc', 'Act_Acc', 'Tot_Ret', 'Tot_Dem']
for k in durations:
    ordered += [f'LP-Acc-{k}-wk', f'Act-Acc-{k}-wk', f'Ret-{k}-wk', f'Dem-{k}-wk']

inv_comparison = pd.DataFrame(data, index=range(1, 53))[ordered]
inv_comparison.index.name = 'Week'
inv_comparison = inv_comparison.fillna(0)
print(inv_comparison.to_string())

      Act_Op_Inv  LP_Op_Inv  LP_Acc  Act_Acc  Tot_Ret  Tot_Dem  LP-Acc-1-wk  Act-Acc-1-wk  Ret-1-wk  Dem-1-wk  LP-Acc-4-wk  Act-Acc-4-wk  Ret-4-wk  Dem-4-wk  LP-Acc-8-wk  Act-Acc-8-wk  Ret-8-wk  Dem-8-wk  LP-Acc-16-wk  Act-Acc-16-wk  Ret-16-wk  Dem-16-wk
Week                                                                                                                                                                                                                                                          
1             77       77.0     0.0      0.0      0.0      0.0          0.0           0.0       0.0       0.0          0.0           0.0       0.0       0.0          0.0           0.0       0.0       0.0           0.0            0.0        0.0        0.0
2            102      102.0    26.0     26.0     25.0     26.0          6.0           6.0       1.0       6.0          4.0           4.0      10.0       4.0         10.0          10.0      10.0      10.0           6.0            6.0   